## Objective

- Save and retrieve processed data efficiently inside Dataproc.
- Serve data in a structured way for analysis.
- Use Parquet, Hive, and CSV.


In [1]:
spark = SparkSession.builder \
    .appName('Olist Ecommerce Performance Optimization') \
    .config('spark.executor.memory', '6g') \
    .config('spark.executor.cores', '4') \
    .config('spark.executor.instances', '2') \
    .config('spark.driver.memory', '4g') \
    .config('spark.driver.maxResultSize', '2g') \
    .config('spark.sql.shuffle.partitions', '64') \
    .config('spark.default.parallelism', '64') \
    .config('spark.sql.adaptive.enabled', 'true') \
    .config('spark.sql.adaptive.coalescePartition.enabled', 'true') \
    .config('spark.sql.autoBroadcastJoinThreshold', 20*1024*1024) \
    .config('spark.sql.files.maxPartitionBytes', '64MB') \
    .config('spark.sql.files.openCostInBytes', '2MB') \
    .config('spark.memory.fraction', 0.8) \
    .config('spark.memory.storageFraction', 0.2) \
    .getOrCreate()

26/02/15 09:57:16 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [2]:
!hadoop fs -ls /data/olist_proc/

Found 11 items
drwxr-xr-x   - root hadoop          0 2026-02-14 16:13 /data/olist_proc/cleaned_data.parquet
drwxr-xr-x   - root hadoop          0 2026-02-14 16:21 /data/olist_proc/customers_df_cleaned.parquet
drwxr-xr-x   - root hadoop          0 2026-02-15 07:57 /data/olist_proc/full_orders_df_op.parquet
drwxr-xr-x   - root hadoop          0 2026-02-14 16:22 /data/olist_proc/geolocation_df.parquet
drwxr-xr-x   - root hadoop          0 2026-02-14 16:21 /data/olist_proc/order_item_df_cleaned.parquet
drwxr-xr-x   - root hadoop          0 2026-02-14 16:20 /data/olist_proc/orders_df_cleaned.parquet
drwxr-xr-x   - root hadoop          0 2026-02-14 16:21 /data/olist_proc/payments_df.parquet
drwxr-xr-x   - root hadoop          0 2026-02-14 16:37 /data/olist_proc/product_df_cleaned.parquet
drwxr-xr-x   - root hadoop          0 2026-02-14 16:19 /data/olist_proc/products_df_cleaned.parquet
drwxr-xr-x   - root hadoop          0 2026-02-14 16:20 /data/olist_proc/reviews_df_cleaned.parquet
drwxr-xr

In [3]:
full_orders_df_op = spark.read.parquet('/data/olist_proc/full_orders_df_op.parquet')

In [4]:
full_orders_df_op.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (null

In [17]:
!hadoop fs -mkdir /data/olist/processed

In [19]:
!hadoop fs -ls /data/olist/processed

In [20]:
# save as Parquet in hdfs

full_orders_df_op.write.mode('overwrite').parquet('/data/olist/processed/full_orders_df_op.parquet')

In [21]:
!hadoop fs -ls /data/olist/processed/

Found 1 items
drwxr-xr-x   - root hadoop          0 2026-02-15 10:26 /data/olist/processed/full_orders_df_op.parquet


In [23]:
full_orders_df_op = spark.read.parquet('/data/olist/processed/full_orders_df_op.parquet')

In [24]:
full_orders_df_op.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (null

In [26]:
# Save as a parquet in Google cloud Storage

full_orders_df_op.write.mode('overwrite').parquet('gs://dataproc-staging-us-central1-1062396121388-mfsgnuyn/temp_data')

In [29]:
# stored in google cloud storage 
# first one staging
# then in temp_data
# we have created new temp_data to stored data 
# but to reduce cost we have delete that temp_data

In [27]:
# Save data as sql hive

In [28]:
full_orders_df_op.write.mode('overwrite').saveAsTable('full_orders_df_op')

ivysettings.xml file not found in HIVE_HOME or HIVE_CONF_DIR,/etc/hive/conf.dist/ivysettings.xml will be used
26/02/15 10:40:39 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.


In [30]:
# Save data as a csv

In [31]:
full_orders_df_op.write.mode('overwrite').option('header','true').csv('/data/olist/processed/full_orders_df_op.csv')

In [33]:
!hadoop fs -ls /data/olist/processed/

26/02/15 10:53:14 WARN YarnAllocatorNodeHealthTracker: No available nodes reported, please check Resource Manager.


Found 2 items
drwxr-xr-x   - root hadoop          0 2026-02-15 10:51 /data/olist/processed/full_orders_df_op.csv
drwxr-xr-x   - root hadoop          0 2026-02-15 10:26 /data/olist/processed/full_orders_df_op.parquet


26/02/15 10:53:17 WARN YarnAllocatorNodeHealthTracker: No available nodes reported, please check Resource Manager.
26/02/15 10:53:20 WARN YarnAllocatorNodeHealthTracker: No available nodes reported, please check Resource Manager.
26/02/15 10:53:23 WARN YarnAllocatorNodeHealthTracker: No available nodes reported, please check Resource Manager.
26/02/15 10:53:26 WARN YarnAllocatorNodeHealthTracker: No available nodes reported, please check Resource Manager.


In [34]:
spark.stop()

26/02/15 10:53:28 WARN YarnAllocatorNodeHealthTracker: No available nodes reported, please check Resource Manager.


In [1]:
spark.stop()

26/02/15 13:13:11 WARN YarnAllocatorNodeHealthTracker: No available nodes reported, please check Resource Manager.
